In [ ]:
import numpy as np

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

1. Implement logistic regression / linear regression from scratch

In [ ]:
class LinearRegressionGD:
    """
    Linear Regression using Gradient Descent.
    
    Parameters:
        lr: Learning rate
        epochs: Number of training epochs
        batch_size: Mini-batch size (None for full batch GD)
        seed: Random seed for reproducibility
    """
    
    def __init__(self, lr=1e-1, epochs=500, batch_size=None, seed=0):
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.seed = seed
        self.w = None
        self.b = None
    
    def fit(self, X, y):
        """
        Fit the linear regression model.
        
        Args:
            X: (N, D) float tensor - features
            y: (N,) or (N, 1) float tensor - targets
        
        Returns:
            self
        """
        torch.manual_seed(self.seed)
        X = X.float()
        y = y.float().view(-1, 1)
        N, D = X.shape
        
        self.w = torch.zeros(D, 1, requires_grad=True)
        self.b = torch.zeros(1, requires_grad=True)
        
        batch_size = self.batch_size if self.batch_size is not None else N
        loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True)
        
        for _ in range(self.epochs):
            for Xb, yb in loader:
                pred = Xb @ self.w + self.b       # (B, 1)
                loss = ((pred - yb) ** 2).mean()  # MSE
                
                loss.backward()
                with torch.no_grad():
                    self.w -= self.lr * self.w.grad
                    self.b -= self.lr * self.b.grad
                    self.w.grad.zero_()
                    self.b.grad.zero_()
        
        # Detach after training
        self.w = self.w.detach()
        self.b = self.b.detach()
        
        return self
    
    def predict(self, X):
        """
        Predict using the linear model.
        
        Args:
            X: (N, D) float tensor - features
        
        Returns:
            (N,) tensor of predictions
        """
        if self.w is None:
            raise RuntimeError("Model not fitted. Call fit() first.")
        
        X = X.float()
        return (X @ self.w + self.b).squeeze(-1)


In [ ]:
# the fit function can be replaced with the following:
def fit_linear_regression_torch(X, y, lr=1e-1, epochs=500, batch_size=None):
    X = X.float()
    y = y.float().view(-1, 1)
    N, D = X.shape

    model = nn.Linear(D, 1)
    opt = torch.optim.SGD(model.parameters(), lr=lr)

    if batch_size is None:
        batch_size = N

    loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True)

    for _ in range(epochs):
        for Xb, yb in loader:
            pred = model(Xb)
            loss = F.mse_loss(pred, yb)

            opt.zero_grad()
            loss.backward()
            opt.step()

    return model


In [ ]:
class LogisticRegressionGD:
    """
    Logistic Regression using Gradient Descent.
    
    Parameters:
        lr: Learning rate
        epochs: Number of training epochs
        batch_size: Mini-batch size (None for full batch GD)
        l2: L2 regularization strength
        seed: Random seed for reproducibility
        threshold: Classification threshold for predict()
    """
    
    def __init__(self, lr=1e-1, epochs=500, batch_size=None, l2=0.0, seed=0, threshold=0.5):
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.l2 = l2
        self.seed = seed
        self.threshold = threshold
        self.w = None
        self.b = None
    
    def fit(self, X, y):
        """
        Fit the logistic regression model.
        
        Args:
            X: (N, D) float tensor - features
            y: (N,) tensor with values in {0, 1} - binary labels
        
        Returns:
            self
        """
        torch.manual_seed(self.seed)
        X = X.float()
        y = y.float().view(-1, 1)
        N, D = X.shape
        
        self.w = torch.zeros(D, 1, requires_grad=True)
        self.b = torch.zeros(1, requires_grad=True)
        
        batch_size = self.batch_size if self.batch_size is not None else N
        loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True)
        
        for _ in range(self.epochs):
            for Xb, yb in loader:
                logits = Xb @ self.w + self.b  # (B, 1)
                loss = F.binary_cross_entropy_with_logits(logits, yb)
                
                if self.l2 > 0:
                    loss = loss + 0.5 * self.l2 * (self.w.square().sum())
                
                loss.backward()
                with torch.no_grad():
                    self.w -= self.lr * self.w.grad
                    self.b -= self.lr * self.b.grad
                    self.w.grad.zero_()
                    self.b.grad.zero_()
        
        # Detach after training
        self.w = self.w.detach()
        self.b = self.b.detach()
        
        return self
    
    def predict_proba(self, X):
        """
        Predict class probabilities.
        
        Args:
            X: (N, D) float tensor - features
        
        Returns:
            (N,) tensor of probabilities for the positive class
        """
        if self.w is None:
            raise RuntimeError("Model not fitted. Call fit() first.")
        
        X = X.float()
        logits = X @ self.w + self.b
        return torch.sigmoid(logits).squeeze(-1)
    
    def predict(self, X):
        """
        Predict class labels.
        
        Args:
            X: (N, D) float tensor - features
        
        Returns:
            (N,) tensor of predicted labels {0, 1}
        """
        proba = self.predict_proba(X)
        return (proba >= self.threshold).long()


In [ ]:
# the fit function can be replaced with the following:
def fit_logistic_regression_torch(X, y, lr=1e-1, epochs=500, batch_size=None):
    X = X.float()
    y = y.float().view(-1, 1)
    N, D = X.shape

    model = nn.Linear(D, 1)
    opt = torch.optim.SGD(model.parameters(), lr=lr)

    if batch_size is None:
        batch_size = N

    loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True)

    for _ in range(epochs):
        for Xb, yb in loader:
            logits = model(Xb)
            loss = F.binary_cross_entropy_with_logits(logits, yb)

            opt.zero_grad()
            loss.backward()
            opt.step()

    return model


2. Implement softmax + cross-entropy loss

In [ ]:
def softmax_cross_entropy_from_logits(logits, y):
    """
    logits: (N, C)
    y: (N,) int64 class indices
    returns scalar loss
    """
    # log_softmax is stable (uses log-sum-exp internally)
    log_probs = F.log_softmax(logits, dim=1)                # (N, C)
    loss = -log_probs[torch.arange(logits.size(0)), y].mean()
    return loss

In [ ]:
def grad_softmax_ce_wrt_logits(logits, y):
    """
    returns dlogits: (N, C)
    """
    N, C = logits.shape
    probs = F.softmax(logits, dim=1)                       # (N, C)
    dlogits = probs.clone()
    dlogits[torch.arange(N), y] -= 1.0
    dlogits /= N
    return dlogits


3. Write forward & backward pass for a small neural network

In [ ]:
# Classic implementation
class TwoLayerNet(nn.Module):
    def __init__(self, d_in, d_hidden, d_out):
        super().__init__()
        self.fc1 = nn.Linear(d_in, d_hidden)
        self.fc2 = nn.Linear(d_hidden, d_out)

    def forward(self, x):
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        return x  # logits

In [ ]:
# Training step (manual SGD update, no optimizer):
def train_step_manual_sgd(model, Xb, yb, lr=1e-1):
    logits = model(Xb)
    loss = F.cross_entropy(logits, yb)

    loss.backward()
    with torch.no_grad():
        for p in model.parameters():
            p -= lr * p.grad
            p.grad.zero_()
    return loss.item()

In [ ]:
class TwoLayerNetManual:
    def __init__(self, d_in, d_hidden, d_out, seed=0, weight_scale=1e-2):
        g = torch.Generator().manual_seed(seed)
        self.W1 = (weight_scale * torch.randn(d_in, d_hidden, generator=g)).requires_grad_(False)
        self.b1 = torch.zeros(d_hidden).requires_grad_(False)
        self.W2 = (weight_scale * torch.randn(d_hidden, d_out, generator=g)).requires_grad_(False)
        self.b2 = torch.zeros(d_out).requires_grad_(False)

    def forward(self, X):
        z1 = X @ self.W1 + self.b1          # (N,H)
        a1 = torch.clamp(z1, min=0.0)       # ReLU
        logits = a1 @ self.W2 + self.b2     # (N,C)
        cache = (X, z1, a1, logits)
        return logits, cache

    def loss_and_grads(self, X, y):
        logits, (X, z1, a1, _) = self.forward(X)

        # softmax CE loss (stable)
        log_probs = F.log_softmax(logits, dim=1)
        N = X.size(0)
        loss = -log_probs[torch.arange(N), y].mean()

        # dlogits = (softmax - onehot) / N
        probs = torch.exp(log_probs)  # softmax
        dlogits = probs
        dlogits = dlogits.clone()
        dlogits[torch.arange(N), y] -= 1.0
        dlogits /= N

        # backprop
        dW2 = a1.t() @ dlogits                        # (H,C)
        db2 = dlogits.sum(dim=0)                      # (C,)
        da1 = dlogits @ self.W2.t()                   # (N,H)

        dz1 = da1.clone()
        dz1[z1 <= 0] = 0.0                            # ReLU backward

        dW1 = X.t() @ dz1                             # (D,H)
        db1 = dz1.sum(dim=0)                          # (H,)

        grads = {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}
        return loss, grads

    def step(self, grads, lr):
        self.W1 -= lr * grads["W1"]
        self.b1 -= lr * grads["b1"]
        self.W2 -= lr * grads["W2"]
        self.b2 -= lr * grads["b2"]

4. Implement gradient descent (batch / mini-batch)

In [ ]:
import numpy as np

def iterate_minibatches(X, y, batch_size, shuffle=True, seed=None):
    X = np.asarray(X)
    y = np.asarray(y)
    n = X.shape[0]
    idx = np.arange(n)
    if shuffle:
        rng = np.random.default_rng(seed)
        rng.shuffle(idx)

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch_idx = idx[start:end]
        yield X[batch_idx], y[batch_idx]

def gradient_descent(train_step_fn, X, y, lr=1e-2, epochs=10, batch_size=None, seed=0):
    """
    train_step_fn(Xb, yb, lr) should do: compute loss/grads and update parameters.
    If batch_size is None -> batch GD. Else -> mini-batch GD.
    """
    if batch_size is None:
        # Batch GD
        for _ in range(epochs):
            train_step_fn(X, y, lr)
    else:
        # Mini-batch GD
        for ep in range(epochs):
            for Xb, yb in iterate_minibatches(X, y, batch_size, shuffle=True, seed=seed + ep):
                train_step_fn(Xb, yb, lr)


In [ ]:
def make_train_step(net):
    def step(Xb, yb, lr):
        loss, grads = net.loss_and_grads(Xb, yb)
        net.step(grads, lr)
        return loss
    return step

In [ ]:
def train_with_dataloader(
    model,
    X, y,
    loss_fn,
    lr=1e-2,
    epochs=10,
    batch_size=None,
    shuffle=True,
    use_optimizer=False
):
    X = X.float()
    y = y.long() if y.dtype != torch.long else y

    N = X.size(0)
    if batch_size is None:
        batch_size = N  # batch GD

    loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=shuffle)

    if use_optimizer:
        opt = torch.optim.SGD(model.parameters(), lr=lr)

    for _ in range(epochs):
        for Xb, yb in loader:
            preds = model(Xb)
            loss = loss_fn(preds, yb)

            if use_optimizer:
                opt.zero_grad()
                loss.backward()
                opt.step()
            else:
                loss.backward()
                with torch.no_grad():
                    for p in model.parameters():
                        p -= lr * p.grad
                        p.grad.zero_()

    return model


In [ ]:
# For classification MLP:
model = TwoLayerNet(d_in=10, d_hidden=32, d_out=5)
train_with_dataloader(model, X, y, loss_fn=F.cross_entropy, lr=0.1, epochs=50, batch_size=64)

# For linear regression:
lin = nn.Linear(10, 1)
train_with_dataloader(lin, X, y_float.view(-1,1), loss_fn=F.mse_loss, lr=0.1, epochs=200, batch_size=None)

5. implement k-means clustering

In [ ]:
import numpy as np

def kmeans(X, k, max_iters=100, tol=1e-4, seed=0):
    """
    X: (N, D) ndarray
    returns: centroids (k, D), labels (N,)
    """
    X = np.asarray(X, dtype=float)
    N, D = X.shape
    rng = np.random.default_rng(seed)

    # init centroids by sampling points
    centroids = X[rng.choice(N, size=k, replace=False)].copy()
    labels = np.zeros(N, dtype=int)

    for _ in range(max_iters):
        # assign: compute squared distances to each centroid
        # dists: (N, k)
        dists = ((X[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
        new_labels = np.argmin(dists, axis=1)

        # update centroids
        new_centroids = centroids.copy()
        for j in range(k):
            pts = X[new_labels == j]
            if len(pts) == 0:
                # empty cluster: re-seed to a random point
                new_centroids[j] = X[rng.integers(0, N)]
            else:
                new_centroids[j] = pts.mean(axis=0)

        # convergence
        shift = np.linalg.norm(new_centroids - centroids)
        centroids, labels = new_centroids, new_labels
        if shift < tol:
            break

    return centroids, labels


6. Compute precision, recall, F1 score, auc

In [ ]:
import numpy as np

def precision_recall_f1(y_true, y_pred):
    """
    y_true, y_pred: arrays of 0/1
    returns (precision, recall, f1)
    """
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

In [ ]:
import numpy as np

def roc_auc_score_from_scratch(y_true, y_score):
    """
    Computes ROC-AUC using the rank statistic (equivalent to Mann–Whitney U).
    y_true: (N,) in {0,1}
    y_score: (N,) real-valued scores (higher => more positive)
    """
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    N = len(y_true)

    # ranks with tie handling via average rank
    order = np.argsort(y_score)
    ranks = np.empty(N, dtype=float)
    ranks[order] = np.arange(1, N + 1)  # 1..N

    # fix ties: average ranks for equal scores
    sorted_scores = y_score[order]
    i = 0
    while i < N:
        j = i
        while j + 1 < N and sorted_scores[j + 1] == sorted_scores[i]:
            j += 1
        if j > i:
            avg_rank = ranks[order[i:j+1]].mean()
            ranks[order[i:j+1]] = avg_rank
        i = j + 1

    pos = (y_true == 1)
    n_pos = pos.sum()
    n_neg = N - n_pos
    if n_pos == 0 or n_neg == 0:
        return float("nan")

    sum_ranks_pos = ranks[pos].sum()
    # Mann–Whitney U for positives:
    U = sum_ranks_pos - n_pos * (n_pos + 1) / 2.0
    auc = U / (n_pos * n_neg)
    return auc


7. Train / Predict loop for sklearn models

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

def train_eval_sklearn(model, X_train, y_train, X_val, y_val):
    model.fit(X_train, y_train)

    # class prediction
    y_pred = model.predict(X_val)

    # score for AUC if possible
    y_score = None
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_val)[:, 1]
    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(X_val)

    metrics = {
        "accuracy": accuracy_score(y_val, y_pred),
        "precision": precision_score(y_val, y_pred, zero_division=0),
        "recall": recall_score(y_val, y_pred, zero_division=0),
    }
    if y_score is not None:
        metrics["auc"] = roc_auc_score(y_val, y_score)

    return model, metrics


8. Cross-validation logic (from scratch-ish)

In [ ]:
# K-fold CV (manual) for sklearn model
import numpy as np
from sklearn.base import clone

def kfold_indices(n, k, seed=0, shuffle=True):
    idx = np.arange(n)
    if shuffle:
        rng = np.random.default_rng(seed)
        rng.shuffle(idx)
    folds = np.array_split(idx, k)
    return folds

def cross_val_score_manual(model, X, y, k=5, scoring="accuracy", seed=0):
    """
    scoring: 'accuracy' or 'auc'
    """
    X = np.asarray(X)
    y = np.asarray(y)
    n = len(y)

    folds = kfold_indices(n, k, seed=seed, shuffle=True)
    scores = []

    for i in range(k):
        val_idx = folds[i]
        train_idx = np.concatenate([folds[j] for j in range(k) if j != i])

        X_train, y_train = X[train_idx], y[train_idx]
        X_val, y_val = X[val_idx], y[val_idx]

        m = clone(model)
        m.fit(X_train, y_train)

        if scoring == "accuracy":
            y_pred = m.predict(X_val)
            scores.append((y_pred == y_val).mean())

        elif scoring == "auc":
            if hasattr(m, "predict_proba"):
                y_score = m.predict_proba(X_val)[:, 1]
            else:
                y_score = m.decision_function(X_val)
            scores.append(roc_auc_score(y_val, y_score))

        else:
            raise ValueError("Unsupported scoring")

    return float(np.mean(scores)), scores


In [ ]:
# If they want “proper sklearn way”
from sklearn.model_selection import KFold, cross_val_score

cv = KFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(model, X, y, cv=cv, scoring="roc_auc")
print(scores.mean(), scores)
